In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append('..')   

import pandas as pd
from src.data_validation import validate_transactions, validate_customer

customers = pd.read_excel('../data/raw/FinTrust_Customer_Data.xlsx')
transactions = pd.read_excel('../data/raw/FinTrust_Transaction_Data.xlsx')

print(validate_transactions(transactions))
print(validate_customer(customers))

{'missing_device_type': np.int64(96), 'missing_location': np.int64(96), 'duplicate_transaction_ids': np.int64(0), 'unexpected_amount': np.int64(0), 'unexpected_risk_values': np.int64(0)}
{'missing_values': np.int64(0), 'duplicate_customer_ids': np.int64(0), 'unexpected_gender': np.int64(0), 'unexpected_income_band': np.int64(0)}


In [3]:
from src.preprocessing import preprocess_transactions, preprocess_customers

processed = preprocess_transactions(transactions, mode='train')
processed_customers = preprocess_customers(customers)

print(processed.shape)
print(processed_customers.shape)

(11812, 23)
(1500, 24)


In [4]:
from src.preprocessing import merge_data, select_model_columns

merged = merge_data(processed, processed_customers)
model_data = select_model_columns(merged)

print(merged.shape)
print(model_data.shape)

(11812, 46)
(11812, 40)


In [5]:
from sklearn.metrics import classification_report
from src.model import time_based_split, prepare_xy, train_model, predict_labels

train_data, test_data = time_based_split(merged)
X_train, y_train = prepare_xy(train_data)
X_test, y_test = prepare_xy(test_data)

model, scaler = train_model(X_train, y_train)
predictions = predict_labels(model, scaler, X_test)

print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

           0       0.81      0.99      0.89      1906
           1       0.45      0.04      0.07       457

    accuracy                           0.80      2363
   macro avg       0.63      0.51      0.48      2363
weighted avg       0.74      0.80      0.73      2363

